In [8]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

# os.chdir(module_path)
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /home/fre.gilad/source/AgentDac-AGL/AgentDaC/notebooks


In [9]:
import socket

def find_free_port(host: str = "127.0.0.1") -> int:
    """Ask the OS for a free ephemeral port, then release it."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind((host, 0))
        return s.getsockname()[1]

host = "127.0.0.1"
port = find_free_port(host)   # e.g. 51734
model = "Qwen/Qwen3-4B-Instruct-2507"

print(f"vLLM will use {host}:{port}")


vLLM will use 127.0.0.1:49993


In [10]:
import shlex

vllm_serve = [
    "vllm",
    "serve",
    model,
    "--host",
    host,
    "--port",
    str(port),
    "--dtype",
    "bfloat16",
    "--max_model_len",
    "14336",
    "--max_num_seqs",
    "1024",
    "--enable_chunked_prefill",
    "--max_num_batched_tokens",
    "8192",
    "--enable_prefix_caching",
    "--logprobs_mode",
    "processed_logprobs",
    "--gpu_memory_utilization",
    "0.9",
    "--disable_log_stats",
    "--tensor_parallel_size",
    "1",
    "--seed",
    "0",
    "--override_generation_config",
    '{"temperature": 1.0, "top_k": 20, "top_p": 0.95, "repetition_penalty": 1.0, "max_new_tokens": 12288}',
    "--generation_config",
    "auto",
    "--additional_config",
    '{"gdn_prefill_backend": "triton"}',
]

def print_vllm_serve_command():
    print("vllm serve command:")
    print(shlex.join(vllm_serve))
    
print_vllm_serve_command()

vllm serve command:
vllm serve Qwen/Qwen3-4B-Instruct-2507 --host 127.0.0.1 --port 49993 --dtype bfloat16 --max_model_len 14336 --max_num_seqs 1024 --enable_chunked_prefill --max_num_batched_tokens 8192 --enable_prefix_caching --logprobs_mode processed_logprobs --gpu_memory_utilization 0.9 --disable_log_stats --tensor_parallel_size 1 --seed 0 --override_generation_config '{"temperature": 1.0, "top_k": 20, "top_p": 0.95, "repetition_penalty": 1.0, "max_new_tokens": 12288}' --generation_config auto --additional_config '{"gdn_prefill_backend": "triton"}'


In [11]:
from src.inference import OAIClient

# host = "127.0.0.1"
# port = "34803"

base_url = f"http://{host}:{port}/v1"

client = OAIClient(model_name=model, base_url=base_url)

In [ ]:
from src.agents import PersistentAgent, MarkerAgent
from src.configs import PromptConfig, DecompConfig


prompt_config = PromptConfig(
    mode="path",
    system_root="../config_files/prompts/perst/v5_root.txt",
    system_inter="../config_files/prompts/perst/v5_root.txt",
    system_leaf="../config_files/prompts/perst/v5_leaf.txt",
    tasks_depleted=None,
)

decomp_config = DecompConfig(
    max_depth=1,
    max_tasks=4,
    max_rounds=5,
)



In [22]:
from experiments.chess_perst.format import format_prompt

fens = [
    "r4rk1/5ppp/pQ1b1q2/8/1P6/P4N1P/5PP1/3R1RK1 b - - 2 25",
    "r5k1/2n5/5r1p/p1pP1N2/PpP2p2/1P1Q4/3B1q1N/7K w - - 0 33",
    "R4bk1/5rP1/5pR1/8/2r5/p5P1/6K1/1q6 w - - 0 42",
    "2r4k/p5pp/1p2q3/2p3Q1/8/7P/PB3PP1/6K1 w - - 0 30",
    "2Q5/1p3kp1/3p2p1/3Pp3/2N1n3/P7/KQq5/8 b - - 1 33",
    "1rr5/pbRp3p/1p1nkpp1/4p3/1B2P1P1/3B1P1P/PP2K3/2R5 w - - 8 25",
    "Q7/4ppbk/6pp/5q2/P3n3/4PNBP/1r3PP1/3R2K1 b - - 0 31",
    "1r3rk1/pbp3bp/6p1/2qN1p2/1P2B3/5Q2/P1P2PPP/1R3RK1 b - - 0 16",
]


samples = [{"fen": fen} for fen in fens]
prompts = [format_prompt(sample) for sample in samples]

In [23]:
kwargs = {
    "temperature": 1.0,
    "top_p": 0.95,
    "extra_body": {
        "chat_template_kwargs": {"enable_thinking": False},
        "min_tokens": 5,
    },
}


for i, prompt in enumerate(prompts[:2]):
    print("--------------------"*10)
    print(f"Prompt {i + 1}:\n{prompt}\n")
    
    message = {
        "role": "user",
        "content": prompt
    }
        
    agent = PersistentAgent(
        client=client,
        prompt_config=prompt_config,
        decomp_config=decomp_config,
        additional_histories=True,
    )
    
    response = await agent.chat(prompt=message, verbose=True, **kwargs)

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Prompt 1:
Task:
You are given a chess position. Find the best legal move for the side to move. Important: don't reason too long, you have token limits.Use sub-tasks to break down the problem if needed and break down the reasoning into smaller steps.

Current FEN:
r4rk1/5ppp/pQ1b1q2/8/1P6/P4N1P/5PP1/3R1RK1 b - - 2 25

Side to move:
Black

Board, White at bottom (uppercase = White, lowercase = Black, '.' = empty square):
8  r . . . . r k .
7  . . . . . p p p
6  p Q . b . q . .
5  . . . . . . . .
4  . P . . . . . .
3  P . . . . N . P
2  . . . . . P P .
1  . . . R . R K .
   a b c d e f g h

Legal moves (UCI):
a6a5 a8a7 a8b8 a8c8 a8d8 a8e8 d6b4 d6b8 d6c5 d6c7 d6e5 d6e7 d6f4 d6g3 d6h2 f6a1 f6b2 f6c3 f6d4 f6d8 f6e5 f6e6 f6e7 f6f3 f6f4 f6f5 f6g5 f6g6 f6h4 f6h6 f8b8 f8c8 f8d8 f8e8 g7g5 g7g6 g8h8 

Role:
ASSISTANT
Content:
Action: think
Text: The current FEN is r4rk1/5ppp/pQ1b1q2/8/1P6/P4N1P/5PP1/3R1RK1 b - - 2 25, and Black is to move. I need to find the best legal move. First, I will analyze the position for immediate threats, piece activity, and potential checks or captures. Black has a queen on c2, a rook on h8, and a knight on g6. White has a pawn on d4, a queen on b1, and a knight on f3. I need to evaluate if there's a forced move or tactical sequence. I notice that Black's king is on d8, which is safe. A key point is the queen on c2 and the black bishop on b1—this might be involved in a potential attack. However, the position is complex, and I should check for immediate threats or pawn advances. Black may aim to exploit weaknesses in White's center or attack the queen. The best move is likely one that increases pressure or defends key squares. After reviewing, I see that a potential move is d6c7 (moving the knight from d6 to c7), which could support the center and prepare 